# Prática — Feature Engineering (Aula 3)
## Transformando as bases de eventos em uma tabela de modelagem

**O que vocês já têm:**
- O cadastro de colaboradores (`FtFuncionarioRH_amostra.csv`, Aula 2)
- O alvo já derivado (`alvo_derivado.csv`) — `pediu_para_sair`, construído a partir de `cIniciativaDemissao`
- As 5 bases de eventos da Aula 1 (`FtAbsenteismoMensalRH`, `FtAcidentesRH`, `FtHoraExtraRH`, `FtHorasIrregularesRH`, `FtMovimentoSalarialRH`)

**O que é novo hoje:**
- `datas_referencia.csv` — uma **data de referência por colaborador**. Ao construir qualquer métrica agregada, usem **apenas eventos até essa data** (inclusive). Isso não é opcional — é a mesma regra de janela temporal que vimos no material de apoio.

**Objetivo:** produzir uma única tabela, **uma linha por `nIdPessoa`**, juntando cadastro + alvo + métricas agregadas das 5 bases de eventos.

Dois exemplos abaixo já vêm resolvidos, como referência de padrão. As demais bases ficam para vocês.

## 0. Carregando os dados

In [8]:
import pandas as pd
pd.set_option('display.max_columns', None)

# Caminhos relativos ao diretório do notebook
# Notebook está em: aula3_material_recebido/notebook_pratica/
# Precisa voltar para: aula1_material_recebido/
A1 = "../../../aula1/"
A2 = "../../../aula2/"
PRATICA = "./"

cadastro = pd.read_csv(A2 + "FtFuncionarioRH_amostra.csv", sep=";")
alvo = pd.read_csv(PRATICA + "alvo_derivado.csv", sep=";")
datas_ref = pd.read_csv(PRATICA + "datas_referencia.csv", sep=";")
datas_ref['data_referencia'] = pd.to_datetime(datas_ref['data_referencia'])

print(cadastro.shape, alvo.shape, datas_ref.shape)
datas_ref.head()

FileNotFoundError: [Errno 2] No such file or directory: '../../../aula2/FtFuncionarioRH_amostra.csv'

## 1. Função utilitária: filtrar eventos pela data de referência

Já está pronta — vocês vão usá-la em todas as bases.

In [ ]:
cutoff_map = datas_ref.set_index('nIdPessoa')['data_referencia']

def filtra_por_data(df, col_data, dayfirst=True):
    """Mantem apenas linhas com data <= data de referencia do respectivo colaborador."""
    df = df.copy()
    df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)
    df['__ref'] = df['nIdPessoa'].map(cutoff_map)
    df = df[df['__data'].notna() & df['__ref'].notna() & (df['__data'] <= df['__ref'])]
    return df.drop(columns=['__data', '__ref'])

def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce")

## 2. Exemplo resolvido — Acidentes

Coluna de data: `dDataAcidente`. Vamos gerar 3 métricas: número de eventos, quantos tiveram afastamento, e total de dias perdidos.

In [ ]:
acid = pd.read_csv(A1 + "FtAcidentesRH.csv", sep=";")
acid = filtra_por_data(acid, 'dDataAcidente')

acid['nComAfastamento_n'] = to_num(acid['nComAfastamento'])
acid['nDiasPerdidos_n'] = to_num(acid['nDiasPerdidos'])

agg_acid = acid.groupby('nIdPessoa').agg(
    acidentes_eventos=('nComAfastamento_n', 'count'),
    acidentes_com_afastamento=('nComAfastamento_n', 'sum'),
    acidentes_dias_perdidos=('nDiasPerdidos_n', 'sum')
).reset_index()

print(agg_acid.shape)
agg_acid.head()

## 3. Exemplo resolvido — Movimentação Salarial

Coluna de data: `dMudanca`. Métricas: número de eventos, valor total, e **percentual médio** (média, não soma — percentual não se acumula da mesma forma que valor).

In [ ]:
mov = pd.read_csv(A1 + "FtMovimentoSalarialRH.csv", sep=";")
mov = filtra_por_data(mov, 'dMudanca')

mov['nValor_n'] = to_num(mov['nValor'])
mov['nPerc_n'] = to_num(mov['nPerc'])

agg_mov = mov.groupby('nIdPessoa').agg(
    mov_sal_eventos=('nValor_n', 'count'),
    mov_sal_valor_total=('nValor_n', 'sum'),
    mov_sal_perc_medio=('nPerc_n', 'mean')
).reset_index()

print(agg_mov.shape)
agg_mov.head()

## 4. Exercício 1 — Absenteísmo

Coluna de data: `dAnoMes`. Colunas disponíveis: `nQtdeAbsenteismo`, `nHoraPrevista`, `cTipo`, `nIdPessoa`.

Construam uma tabela `agg_abse` com uma linha por `nIdPessoa` e as colunas:
- `abs_eventos` — quantidade de eventos
- `abs_qtd_total` — soma de `nQtdeAbsenteismo`
- `horas_previstas_total` — soma de `nHoraPrevista`

Sigam o mesmo padrão dos exemplos acima (filtrar por data, converter para número, agregar).

In [ ]:
abse = pd.read_csv(A1 + "FtAbsenteismoMensalRH.csv", sep=";")

# TODO: filtrar por data de referencia (coluna dAnoMes)


# TODO: converter as colunas numericas relevantes (lembrem do to_num)


# TODO: agregar por nIdPessoa -> agg_abse (abs_eventos, abs_qtd_total, horas_previstas_total)




## 5. Exercício 2 — Hora Extra

Coluna de data: `dAnoMes`. Colunas disponíveis: `nReferencia`, `nValor`, `nIdPessoa`.

Construam `agg_hext` com:
- `he_eventos` — quantidade de eventos
- `he_referencia_total` — soma de `nReferencia`
- `he_valor_total` — soma de `nValor`

In [ ]:
hext = pd.read_csv(A1 + "FtHoraExtraRH.csv", sep=";")

# TODO: filtrar por data de referencia (coluna dAnoMes)


# TODO: converter as colunas numericas relevantes


# TODO: agregar por nIdPessoa -> agg_hext (he_eventos, he_referencia_total, he_valor_total)




## 6. Exercício 3 — Horas Irregulares

Coluna de data: `dOcorrencia`. Colunas disponíveis: `nMinutosIrregularesExcedidos`, `nMinutosExtras`, `nIdPessoa`.

Construam `agg_hirr` com:
- `hi_eventos` — quantidade de eventos
- `hi_minutos_irregulares` — soma de `nMinutosIrregularesExcedidos`
- `hi_minutos_extras` — soma de `nMinutosExtras`

In [ ]:
hirr = pd.read_csv(A1 + "FtHorasIrregularesRH.csv", sep=";", low_memory=False)

# TODO: filtrar por data de referencia (coluna dOcorrencia)


# TODO: converter as colunas numericas relevantes


# TODO: agregar por nIdPessoa -> agg_hirr (hi_eventos, hi_minutos_irregulares, hi_minutos_extras)




## 7. Juntando tudo

Esta parte já está pronta — só roda depois que `agg_abse`, `agg_hext` e `agg_hirr` estiverem criados acima.

In [ ]:
df = cadastro.merge(alvo, on='nIdPessoa', how='left')
for agg in [agg_abse, agg_acid, agg_hext, agg_hirr, agg_mov]:
    df = df.merge(agg, on='nIdPessoa', how='left')

event_cols = ['abs_eventos','abs_qtd_total','horas_previstas_total','acidentes_eventos',
    'acidentes_com_afastamento','acidentes_dias_perdidos','he_eventos','he_referencia_total',
    'he_valor_total','hi_eventos','hi_minutos_irregulares','hi_minutos_extras',
    'mov_sal_eventos','mov_sal_valor_total','mov_sal_perc_medio']
df[event_cols] = df[event_cols].fillna(0)
df['pediu_para_sair'] = df['pediu_para_sair'].astype(int)

print("Shape final:", df.shape)
df.head()

## 8. Conferência final

Antes de considerar pronto, respondam:
1. `df.shape[0]` bate com o número de colaboradores do cadastro?
2. A proporção de `pediu_para_sair` continua parecida com o que vimos na Aula 2?
3. Alguma das suas 3 colunas de `abs_eventos`, `he_eventos` ou `hi_eventos` parece grande ou pequena demais? Por quê?

In [ ]:
print(df['pediu_para_sair'].value_counts(normalize=True))
print()
print(df[['abs_eventos','he_eventos','hi_eventos']].describe())